# 01 - Oracle Database 26ai + OCI Generative AI using the OCI SDK

This notebook follows the written guide: `../OracleDB-GenAIOCI-Connection.md`.

The goal is to show how easy it is to use OCI Generative AI in a small database RAG pattern. Setup and database plumbing stay compact, while the notebook makes the two main GenAI service calls easy to see:

1. Turn document chunks and questions into embeddings with `genai_client.embed_text(...)`.
2. Search those vectors in Oracle Database 26ai.
3. Send the retrieved context to an OCI chat model with `genai_client.chat(...)`.

Official references:

- OCI Python SDK Generative AI Inference examples: https://docs.oracle.com/en-us/iaas/tools/python-sdk-examples/latest/generativeaiinference/
- OCI Generative AI API key authentication: https://docs.oracle.com/en-us/iaas/Content/generative-ai/setup-oci-api-auth.htm
- Oracle AI Vector Search `VECTOR_DISTANCE`: https://docs.oracle.com/en/database/oracle/oracle-database/26/vecse/vector_distance.html
- python-oracledb documentation: https://python-oracledb.readthedocs.io/


Copyright (c) 2026 Oracle and/or its affiliates. Licensed under the Universal Permissive License (UPL), Version 1.0.


## What This Demo Builds

This notebook answers questions over a tiny set of example documents stored in Oracle Database.

```text
User question
    -> OCI embedding model
    -> Oracle Database vector search
    -> top matching document chunks
    -> OCI chat model
    -> grounded answer with sources
```

This is the same pattern as the written guide. The only addition is that this notebook creates and seeds a small example table so the flow is runnable from a clean database schema.

When presenting, focus on the two OCI Generative AI calls: one creates embeddings, and one creates the final grounded answer. The database setup exists to support that flow.


## Install Python Packages

Install dependencies once from the `files` directory by running `uv sync` before opening this notebook.


In [ ]:
# Dependencies are managed by uv. Run `uv sync` from the files directory before opening this notebook.

## Step 1 - Load Workshop Setup

This reuses the same authentication and client setup from notebook `00`.


In [ ]:
from IPython.display import Markdown, display
from pathlib import Path
import os
import sys

import pandas as pd

# Make the repo-level helper package importable from this notebook.
for directory in [Path.cwd(), *Path.cwd().parents]:
    if (directory / "workshop_helpers").exists():
        sys.path.insert(0, str(directory))
        break
else:
    raise FileNotFoundError("Could not find the workshop_helpers folder.")

from workshop_helpers.oci_genai_helpers import create_genai_client, load_workshop_env, mask

# Load .env and create the same OCI GenAI client pattern used in the written guide.
env_path = load_workshop_env()
genai_client = create_genai_client()

display(Markdown(f"""
**OCI Setup Ready**

| Setting | Value |
|---|---|
| `.env` file | `{env_path}` |
| GenAI endpoint | `{os.getenv('GENAI_ENDPOINT')}` |
| Chat model | `{os.getenv('CHAT_MODEL_ID')}` |
| Embedding model | `{os.getenv('EMBED_MODEL_ID')}` |
| Chat request format | `{os.getenv('CHAT_REQUEST_FORMAT', 'COHERE')}` |
"""))

## Step 2 - Check Database Settings

The written guide expects these database values to be available as environment variables:

- `DB_USER`
- `DB_PASSWORD`
- `DB_DSN`

This notebook also uses `DB_DOCS_TABLE`, defaulting to `WORKSHOP_GENAI_DOCS`.


In [ ]:
# Check that all required database settings are present before opening a connection.
required_db_vars = ["DB_USER", "DB_PASSWORD", "DB_DSN"]
optional_db_vars = ["TNS_ADMIN", "DB_WALLET_PASSWORD", "DB_DOCS_TABLE"]

rows = []
missing = []
for name in required_db_vars:
    value = os.getenv(name)
    status = "OK" if value and "replace" not in value.lower() else "MISSING"
    rows.append((name, status, mask(value, 24) if name == "DB_PASSWORD" else value))
    if status == "MISSING":
        missing.append(name)

for name in optional_db_vars:
    value = os.getenv(name)
    rows.append((name, "SET" if value else "DEFAULT", value or "WORKSHOP_GENAI_DOCS"))

table_rows = "\n".join(f"| `{name}` | {status} | `{value}` |" for name, status, value in rows)
display(Markdown(f"""
**Database Variable Check**

| Variable | Status | Value |
|---|---:|---|
{table_rows}
"""))

if missing:
    raise ValueError(f"Update these values in .env before continuing: {', '.join(missing)}")


## Step 3 - Connect To Oracle Database 26ai

This is the database equivalent of the GenAI client setup: create one connection, test it, then reuse it for the rest of the notebook.


In [ ]:
from workshop_helpers.oracle_rag_helpers import check_db_connection, create_db_connection

# Open a database connection with DB_USER, DB_PASSWORD, and DB_DSN.
conn = create_db_connection()
db_info = check_db_connection(conn)

display(Markdown(f"""
**Database Connection Ready**

| Setting | Value |
|---|---|
| Database | `{db_info['db_name']}` |
| Service | `{db_info['service_name']}` |
| Current schema | `{db_info['current_schema']}` |
"""))

## Step 4 - Create A Small Vector Table

The written guide searches a table named `docs` with these logical fields:

```sql
SELECT doc_id, chunk_text, source
FROM docs
ORDER BY VECTOR_DISTANCE(embedding, :qv, COSINE)
FETCH FIRST :k ROWS ONLY
```

Illustrative table shape after we create it:

| doc_id | chunk_text | source | embedding |
|---:|---|---|---|
| 1 | `OCI Generative AI is a fully managed Oracle Cloud service...` | `oci-genai-overview.md` | `[0.0123, -0.0456, ..., 0.0789]` |

Each row stores one text chunk, where it came from, and the vector that represents its meaning.

For this example, we create a similarly shaped table. The vector column is declared as `VECTOR` to keep the setup simple across embedding dimensions.


In [ ]:
from workshop_helpers.oracle_rag_helpers import docs_table_name, ensure_docs_table

# Create the example table if it does not already exist.
table_name = ensure_docs_table(conn)

display(Markdown(f"""
**Vector Table Ready**

Using table: `{table_name}`

```sql
CREATE TABLE {table_name} (
    doc_id NUMBER GENERATED BY DEFAULT ON NULL AS IDENTITY PRIMARY KEY,
    chunk_text VARCHAR2(4000) NOT NULL,
    source VARCHAR2(255) NOT NULL,
    embedding VECTOR
)
```
"""))

## Step 5 - Prepare A Tiny Document Set

A real RAG application would chunk PDFs, web pages, knowledge base articles, or database records. This example uses a few short text chunks so the full flow is easy to inspect.


In [ ]:
# Each row is one searchable chunk. The dataset is intentionally small so the retrieval results are easy to inspect.
sample_documents = [
    {
        "source": "oci-genai-overview.md",
        "chunk_text": "OCI Generative AI is a fully managed Oracle Cloud service for building applications with large language models for chat, summarization, generation, and semantic search.",
    },
    {
        "source": "oracle-db-26ai-vector.md",
        "chunk_text": "Oracle Database 26ai supports vector data and vector similarity search, allowing applications to store embeddings and retrieve semantically related rows.",
    },
    {
        "source": "rag-pattern.md",
        "chunk_text": "Retrieval-Augmented Generation, or RAG, first retrieves relevant context from a trusted data source and then asks a language model to answer using that context.",
    },
    {
        "source": "langchain-oci.md",
        "chunk_text": "The langchain-oci package provides LangChain integrations for OCI Generative AI chat models, completion models, and embedding models.",
    },
    {
        "source": "n8n-gateway.md",
        "chunk_text": "An OpenAI-compatible gateway can let low-code tools such as n8n call OCI Generative AI through familiar chat completion style APIs.",
    },
]

pd.DataFrame(sample_documents)

## Step 6 - Call OCI GenAI To Embed The Documents

This is the first main GenAI moment in the notebook. We start with normal text chunks, send them to the OCI embedding model, and get back vectors.

A vector is just a list of numbers that captures the meaning of the text. Oracle Database stores those vectors so it can search by meaning later.

- document chunks use `input_type="SEARCH_DOCUMENT"`
- user questions later use `input_type="SEARCH_QUERY"`

The important rule is that document embeddings and question embeddings must use the same embedding model.


In [ ]:
import oci
from oci.generative_ai_inference.models import EmbedTextDetails, OnDemandServingMode

from workshop_helpers.oracle_rag_helpers import count_documents, store_embedded_documents

# These are the plain text chunks we want the database to search by meaning.
document_texts_to_embed = [document["chunk_text"] for document in sample_documents]

# This is the visible OCI Generative AI embedding request.
# SEARCH_DOCUMENT tells the model these texts will be stored and searched later.
document_embedding_request = EmbedTextDetails(
    inputs=document_texts_to_embed,
    serving_mode=OnDemandServingMode(
        serving_type="ON_DEMAND",
        model_id=os.environ["EMBED_MODEL_ID"],
    ),
    compartment_id=os.environ["OCI_COMPARTMENT_ID"],
    truncate=EmbedTextDetails.TRUNCATE_END,
    input_type=EmbedTextDetails.INPUT_TYPE_SEARCH_DOCUMENT,
)

# This is the actual OCI Generative AI call that turns text into vectors.
document_embedding_response = genai_client.embed_text(document_embedding_request)
document_embedding_payload = oci.util.to_dict(document_embedding_response.data)

# Read the vector lists out of the OCI response.
document_embeddings = document_embedding_payload.get("embeddings") or [
    item.get("embedding")
    for item in document_embedding_payload.get("data", [])
    if item.get("embedding")
]

if not document_embeddings or len(document_embeddings) != len(sample_documents):
    raise ValueError("OCI returned a different number of embeddings than document chunks.")

# Store the text plus its vector in Oracle Database. The insert details are helper code.
inserted_document_count = store_embedded_documents(
    conn,
    sample_documents,
    document_embeddings,
    table_name=table_name,
)
total_rows = count_documents(conn, table_name=table_name)
first_embedding_preview = ", ".join(f"{value:.4f}" for value in document_embeddings[0][:8])

display(Markdown(f"""
**Documents Embedded And Stored**

| Metric | Value |
|---|---:|
| Plain text chunks sent to OCI | {len(document_texts_to_embed)} |
| Embeddings returned by OCI | {len(document_embeddings)} |
| Vector dimensions | {len(document_embeddings[0])} |
| Inserted in this run | {inserted_document_count} |
| Rows now in `{table_name}` | {total_rows} |
| Embedding model | `{os.getenv('EMBED_MODEL_ID')}` |

**First Vector Preview**

`[{first_embedding_preview}, ...]`
"""))

## Step 7 - Call OCI GenAI To Embed The User Question

This is the same embedding idea, but for the user's question. The question becomes a vector, and Oracle Database uses that vector for similarity search.


In [ ]:
import oci
from oci.generative_ai_inference.models import EmbedTextDetails, OnDemandServingMode

# Change this question to explore how different wording affects retrieval results.
question = "How does Oracle Database help with RAG?"

# SEARCH_QUERY tells the embedding model this text is a user search question.
question_embedding_request = EmbedTextDetails(
    inputs=[question],
    serving_mode=OnDemandServingMode(
        serving_type="ON_DEMAND",
        model_id=os.environ["EMBED_MODEL_ID"],
    ),
    compartment_id=os.environ["OCI_COMPARTMENT_ID"],
    truncate=EmbedTextDetails.TRUNCATE_END,
    input_type=EmbedTextDetails.INPUT_TYPE_SEARCH_QUERY,
)

# This is the actual OCI Generative AI call that turns the question into a vector.
question_embedding_response = genai_client.embed_text(question_embedding_request)
question_embedding_payload = oci.util.to_dict(question_embedding_response.data)

# Read the question vector out of the OCI response.
question_embeddings = question_embedding_payload.get("embeddings") or [
    item.get("embedding")
    for item in question_embedding_payload.get("data", [])
    if item.get("embedding")
]
if not question_embeddings:
    raise ValueError("OCI did not return an embedding for the question.")

query_vector = question_embeddings[0]

display(Markdown(f"""
**Question Embedded**

| Field | Value |
|---|---|
| Question | {question} |
| Vector dimensions | {len(query_vector)} |
| Embedding model | `{os.getenv('EMBED_MODEL_ID')}` |
"""))

## Step 8 - Retrieve Top Matching Chunks From Oracle Database

This is the vector search step from the written guide:

```sql
ORDER BY VECTOR_DISTANCE(embedding, :qv, COSINE)
```

Lower cosine distance means the stored chunk is closer to the question embedding.


In [ ]:
from workshop_helpers.oracle_rag_helpers import retrieve_top_k

# Retrieve the most similar rows from Oracle Database using VECTOR_DISTANCE.
top_k = 3
retrieved_rows = retrieve_top_k(conn, query_vector, k=top_k, table_name=table_name)

retrieved_chunks_markdown = []
for rank, row in enumerate(retrieved_rows, start=1):
    retrieved_chunks_markdown.append(f"""
### Match {rank}

**Source:** `{row['source']}`  
**Document ID:** `{row['doc_id']}`  
**Cosine distance:** `{row['distance']:.4f}`

{row['chunk_text']}
""")

display(Markdown(f"""
**Question**

> {question}

**Top {top_k} Matching Chunks**

{''.join(retrieved_chunks_markdown)}
"""))

## Step 9 - Build The Grounded Prompt

The chat model should not answer from general knowledge. It should answer only from the context retrieved from Oracle Database.


In [ ]:
from workshop_helpers.oracle_rag_helpers import build_grounded_prompt

# Build the same grounded prompt pattern used in the written guide.
grounded_prompt = build_grounded_prompt(question, retrieved_rows)

preview = grounded_prompt[:1800]
if len(grounded_prompt) > len(preview):
    preview += "\n..."

display(Markdown(f"""
**Grounded Prompt Preview**

```text
{preview}
```
"""))


## Step 10 - Call OCI GenAI Chat For A Grounded Answer

This is the second main GenAI moment in the notebook. The chat model receives the user's question plus the matching chunks retrieved from Oracle Database.


In [ ]:
from oci.generative_ai_inference.models import ChatDetails, OnDemandServingMode

from workshop_helpers.oci_genai_helpers import build_chat_request, extract_text_from_chat_response

# The grounded prompt contains the user question plus the matching database chunks.
# build_chat_request hides only the small format difference between COHERE and GENERIC chat models.
chat_model_request = ChatDetails(
    compartment_id=os.environ["OCI_COMPARTMENT_ID"],
    serving_mode=OnDemandServingMode(
        serving_type="ON_DEMAND",
        model_id=os.environ["CHAT_MODEL_ID"],
    ),
    chat_request=build_chat_request(grounded_prompt),
)

# This is the actual OCI Generative AI call that asks the chat model to answer.
chat_model_response = genai_client.chat(chat_model_request)
answer = extract_text_from_chat_response(chat_model_response.data)
sources = "\n".join(f"- `{row['source']}` (doc_id={row['doc_id']})" for row in retrieved_rows)

display(Markdown(f"""
**Question**

> {question}

**Grounded Answer**

{answer}

**Retrieved Sources**

{sources}
"""))


## Try A Different Question

This is the safest interactive cell to rerun. Change only the `question` value and run the cell.

The earlier cells showed the GenAI SDK calls directly. This cell uses compact helper functions to repeat the same flow with less code.


In [ ]:
from workshop_helpers.oracle_rag_helpers import chat_with_grounded_context, embed_query, retrieve_top_k

# Try changing this question and rerunning the cell.
question = "What is the role of an OpenAI-compatible gateway?"

# Same flow as above: embed the question, search the database, then ask the chat model.
top_k = 3
query_vector = embed_query(genai_client, question)
retrieved_rows = retrieve_top_k(conn, query_vector, k=top_k, table_name=table_name)
answer = chat_with_grounded_context(genai_client, question, retrieved_rows)

sources = "\n".join(f"- `{row['source']}` (doc_id={row['doc_id']})" for row in retrieved_rows)
display(Markdown(f"""
**Question**

> {question}

**Answer**

{answer}

**Sources**

{sources}
"""))


## Troubleshooting

| Symptom | Likely cause | What to check |
|---|---|---|
| OCI auth error | API key profile issue | Run notebook `00`; check `OCI_PROFILE` and `OCI_CONFIG_FILE` |
| Model not found | Model unavailable in the region | Check `GENAI_ENDPOINT`, `CHAT_MODEL_ID`, and `EMBED_MODEL_ID` |
| DB connection error | Bad database credentials or DSN | Check `DB_USER`, `DB_PASSWORD`, and `DB_DSN` |
| Table create error | User lacks DDL permission | Use a schema/user that can create the example table |
| Vector insert/search error | Database or driver does not support vector binding | Confirm Oracle Database 26ai and current `oracledb` package |
| Poor retrieval | Query and document embeddings do not match | Use the same `EMBED_MODEL_ID` for documents and questions |
| Generic chat returns no text | Gemini used completion budget for reasoning | Increase `CHAT_MAX_COMPLETION_TOKENS` |

When finished, close the database connection:

```python
conn.close()
```
